If you want to type along with me, head to [this notebook](https://humboldt.cloudbank.2i2c.cloud/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fmlekimchi%2Fdata111_fall26&branch=main&urlpath=tree%2Fdata111_fall26%2Flectures%2Flec24_live.ipynb) instead. If you prefer follow along by executing the cells, stay in this notebook.

# Lecture 24: Interpreting Confidence
v.ekc

The bootstrap again — then the fine print: what "95% confident" actually means, when the method breaks, and how to use an interval to run a hypothesis test. (Slides: KL Lecture 24 deck.)

**Today's sections:**
1. Bootstrap recap — SF compensation
2. A CI for the population mean
3. What confidence means (and doesn't)
4. Try It — using a CI to test a hypothesis

In [ ]:
from datascience import *
%matplotlib inline
path_data = '../../../assets/data/'
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import numpy as np
import warnings
warnings.filterwarnings("ignore")

---
## 1. Bootstrap Recap — SF Compensation

The full pipeline from last time, condensed. Watch for the three bootstrap rules (from the sample, with replacement, same size).

In [ ]:
sf = Table.read_table('san_francisco_2019.csv')
min_salary = 15 * 20 * 50
sf = sf.where('Salary', are.above(min_salary))

In [ ]:
sf_bins = np.arange(0, 726000, 25000)
sf.hist('Total Compensation', bins=sf_bins)

In [ ]:
# Parameter: Median total compensation in the population
def median_comp(t):
    return percentile(50, t.column('Total Compensation'))

median_comp(sf)

In [ ]:
our_sample = sf.sample(400, with_replacement=False)

In [ ]:
our_sample.hist('Total Compensation', bins=sf_bins)

In [ ]:
percentile(50, our_sample.column('Total Compensation') )

# Bootstrap

Sample randomly
 - from the original sample
 - with replacement
 - the same number of times as the original sample size

In [ ]:
# Default behavior of tbl.sample:
# at random with replacement,
# the same number of times as rows of tbl

bootstrap_sample = our_sample.sample()

In [ ]:
bootstrap_sample.hist('Total Compensation', bins=sf_bins)

## Bootstrap Sample Median
This is one estimate of the population median.

In [ ]:
percentile(50, bootstrap_sample.column('Total Compensation'))

In [ ]:
def one_bootstrap_median():
    # draw the bootstrap sample
    resample = our_sample.sample()
    # return the median total compensation in the bootstrap sample
    return percentile(50, resample.column('Total Compensation'))

In [ ]:
one_bootstrap_median()

In [ ]:
# Generate the medians of 1000 bootstrap samples
num_repetitions = 1000
bstrap_medians = make_array()
for i in np.arange(num_repetitions):
    bstrap_medians = np.append(bstrap_medians, one_bootstrap_median())

In [ ]:
resampled_medians = Table().with_column('Bootstrap Sample Median', bstrap_medians)
median_bins=np.arange(120000, 160000, 2000)
resampled_medians.hist(bins = median_bins)

# Plotting parameters; you can ignore this code
parameter_green = '#32CD32'
plots.ylim(-0.000005, 0.00014)
plots.scatter(median_comp(sf), 0, color='red', s=40, zorder=2)
plots.title('Bootstrap Medians and the Parameter (Red Dot)');

## Percentile Method: Middle 95% of the Bootstrap Estimates 

In [ ]:
left = percentile(2.5, bstrap_medians)
right = percentile(97.5, bstrap_medians)

make_array(left, right)

In [ ]:
resampled_medians.hist(bins = median_bins)

# Plotting parameters; you can ignore this code
plots.ylim(-0.000005, 0.00014)
plots.plot(make_array(left, right), make_array(0, 0), color='yellow', lw=3, zorder=1)
%plots.scatter(median_comp(sf), 0, color=parameter_green, s=40, zorder=2);

---
## 2. A Confidence Interval for the Population Mean

Same recipe, new statistic (the mean), new data: 1,174 mother-newborn pairs.

In [ ]:
# Random sample of mother-newborn pairs
births = Table.read_table('baby.csv')
births

In [ ]:
births.hist('Maternal Age')

In [ ]:
# Average age of mothers in the sample
np.average(births.column('Maternal Age'))

### Question
What is the average age of the mothers in the population?

In [ ]:
def one_bootstrap_mean():
    resample = births.sample()
    return np.average(resample.column('Maternal Age'))

In [ ]:
# Generate means from 3000 bootstrap samples
num_repetitions = 3000
bstrap_means = make_array()
for i in np.arange(num_repetitions):
    bstrap_means = np.append(bstrap_means, one_bootstrap_mean())

### Bootstrap Percentile Method for Confidence Interval

The interval of estimates is the "middle 95%" of the bootstrap estimates.

This is called a *95% confidence interval* for the mean age in the population.

In [ ]:
# Get the endpoints of the 95% confidence interval
left = percentile(2.5, bstrap_means)
right = percentile(97.5, bstrap_means)

make_array(left, right)

In [ ]:
resampled_means = Table().with_columns(
    'Bootstrap Sample Mean', bstrap_means
)
resampled_means.hist(bins=15)
plots.plot([left, right], [0, 0], color='yellow', lw=8);

In [ ]:
births.hist('Maternal Age')
plots.plot([left, right], [0, 0], color='yellow', lw=8);

> **What "95% confident" means** (KL deck, slides 17–21): the *process* catches the true parameter in about 95% of samples. It does **not** mean "95% of mothers are in this interval" (look at the last plot — the yellow strip covers almost no one!), and it's not a probability statement about the parameter, which is a fixed number. Confidence lives in the method, not the interval.

> **When not to bootstrap** (deck, slides 15–16): very small samples, extreme percentiles (like the max), or statistics dominated by rare huge values. The bootstrap is powerful, not magic.

## Using the Confidence Interval for Testing Hypotheses

**Null:** The average age of mothers in the population is 25 years; the random sample average is different due to chance.

**Alternative:** The average age of the mothers in the population is not 25 years.

Suppose you use the 5% cutoff for the p-value.

Based on the confidence interval, which hypothesis would you pick?

### Try It 1: CI as a hypothesis test 😊

**Null:** 40% of mothers in the population smoked during pregnancy. **Alternative:** it wasn't 40%.

1. Bootstrap a 95% confidence interval for the proportion of smokers. Is 0.40 inside? What do you conclude?

In [ ]:
# 1. bootstrap CI for the smoking proportion


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Same skeleton as the mean — only the statistic changes. The interval comes out around (0.36, 0.42), which contains 0.40 → the data are consistent with the null.*

```python
# 1.
def one_bootstrap_prop():
    resample = births.sample()
    return np.count_nonzero(resample.column('Maternal Smoker')) / resample.num_rows

props = make_array()
for i in np.arange(3000):
    props = np.append(props, one_bootstrap_prop())

make_array(percentile(2.5, props), percentile(97.5, props))
```

</details>

---
> **Today's takeaway:** a 95% CI is a statement about the method's success rate — and it doubles as a test: if the null value falls outside the interval, reject at the 5% level. Homework 8's interpretation questions come straight from today's fine print.

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### CI ↔ hypothesis test
```python
# reject "parameter = v" at the 5% level  <=>  v outside the 95% CI
```